# Model training and comparison

Trains a fraud model on the gold feature table under the temporal split, evaluates it with fraud-specific metrics, and selects an operational threshold. For the full three-model comparison run `make benchmark`.

**Prerequisites:** `make sample-data`, `make ingest`, and `make features`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
from transaction_risk.models.evaluation import add_positive_probability, evaluate_scored_model
from transaction_risk.models.spark_ml import ModelTrainingConfig, train_model
from transaction_risk.models.split import temporal_split
from transaction_risk.models.thresholding import threshold_for_alert_rate
from transaction_risk.spark.io import read_table

features = read_table(spark, '../data/gold/features')
train_df, validation_df, test_df = temporal_split(features)

model = train_model(train_df, config=ModelTrainingConfig(model_type='logistic_regression'))
scored_validation = add_positive_probability(model.transform(validation_df))
scored_test = add_positive_probability(model.transform(test_df))

In [3]:
# Threshold is selected on the validation window (1% target alert rate), never on test
threshold = threshold_for_alert_rate(scored_validation, alert_rate=0.01)
metrics = evaluate_scored_model(scored_test, top_k=100, threshold=threshold)

print(f'selected threshold: {threshold:.4f}')
for name in ['roc_auc', 'pr_auc', 'precision_at_k', 'recall_at_k', 'precision', 'recall']:
    print(f'{name:>16}: {metrics[name]:.4f}')

selected threshold: 0.9970
         roc_auc: 0.9971
          pr_auc: 0.6623
  precision_at_k: 0.1200
     recall_at_k: 1.0000
       precision: 0.8000
          recall: 0.6667


In [4]:
# Optional: calibrate probabilities on the validation window and inspect the calibration table
from transaction_risk.models.calibration import (
    apply_calibrator,
    brier_score,
    calibration_table,
    fit_platt_calibrator,
)

calibrator = fit_platt_calibrator(scored_validation)
calibrated_test = apply_calibrator(scored_test, calibrator)

print('Brier (raw):       ', round(brier_score(scored_test, 'fraud_probability'), 6))
print('Brier (calibrated):', round(brier_score(calibrated_test, 'calibrated_fraud_probability'), 6))
for row in calibration_table(calibrated_test, 'calibrated_fraud_probability'):
    print(row)
spark.stop()

Brier (raw):        0.005833
Brier (calibrated): 0.004088


{'bin': 0, 'bin_lower': 0.0, 'bin_upper': 0.1, 'count': 744, 'positive_count': 0, 'mean_predicted_probability': 0.00025527359737368354, 'observed_fraud_rate': 0.0}
{'bin': 2, 'bin_lower': 0.2, 'bin_upper': 0.3, 'count': 1, 'positive_count': 0, 'mean_predicted_probability': 0.29853219769150574, 'observed_fraud_rate': 0.0}
{'bin': 4, 'bin_lower': 0.4, 'bin_upper': 0.5, 'count': 1, 'positive_count': 1, 'mean_predicted_probability': 0.41915296296960247, 'observed_fraud_rate': 1.0}
{'bin': 7, 'bin_lower': 0.7, 'bin_upper': 0.8, 'count': 1, 'positive_count': 1, 'mean_predicted_probability': 0.749892356644144, 'observed_fraud_rate': 1.0}
{'bin': 8, 'bin_lower': 0.8, 'bin_upper': 0.9, 'count': 1, 'positive_count': 0, 'mean_predicted_probability': 0.802141369149967, 'observed_fraud_rate': 0.0}
{'bin': 9, 'bin_lower': 0.9, 'bin_upper': 1.0, 'count': 12, 'positive_count': 10, 'mean_predicted_probability': 0.9887885865404574, 'observed_fraud_rate': 0.8333333333333334}
